# TMDB 5000 Movie Dataset으로 장르기반 추천시스템 만들기

* 장르에 있는 텍스트를 벡터화 후 텍스트간 유사도를 구해 가까운 순서대로 정렬
* 유사도 척도: 코사인 유사도

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/haram4th/ablearn/main/tmdb_5000_movies.csv")
data.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [5]:
data['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [6]:
import json

In [7]:
temp = json.loads(data['genres'][0]) # json문자열을 python 객체 (보통 dict)로 변환하는 함수
temp

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [14]:
result = []
for item in temp:
    result.append(item['name'])
result=" ".join(result)  # 리스트를 하나의 문자열로

In [15]:
result

'Action Adventure Fantasy Science Fiction'

In [13]:
data2 = data.copy()

In [16]:
data2['genres'][0] = result
# data2.loc[0, 'genres'] =result  (pandas 3.0인 경우)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19340\63369814.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2['genres'][0] = result


In [17]:
data2

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,220000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",NaN,9367,"[{""id"": 5616, ""name"": ""united states\u2013mexi...",es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,"[{""name"": ""Columbia Pictures"", ""id"": 5}]","[{""iso_3166_1"": ""MX"", ""name"": ""Mexico""}, {""iso...",1992-09-04,2040920,81.0,"[{""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238
4799,9000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",NaN,72766,[],en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,[],[],2011-12-26,0,85.0,[],Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5
4800,0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...",http://www.hallmarkchannel.com/signedsealeddel...,231617,"[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...",en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,"[{""name"": ""Front Street Pictures"", ""id"": 3958}...","[{"

In [19]:
data.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [18]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [20]:
dict_cols = ['genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages']

In [33]:
for col in dict_cols:
    result = []
    for item in data[col]:
#         print(item)
        temp_list = json.loads(item)
        temp_result = []
        for temp_item in temp_list:
            temp_result.append(temp_item['name'])
#         print(" ".join(temp_result))
        result.append(" ".join(temp_result))
    data[col] = result

In [34]:
data

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,Ingenious Film Partners Twentieth Century Fox ...,United States of America United Kingdom,2009-12-10,2787965087,162.0,English Español,Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,Walt Disney Pictures Jerry Bruckheimer Films S...,United States of America,2007-05-19,961000000,169.0,English,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6 bri...,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,Columbia Pictures Danjaq B24,United Kingdom United States of America,2015-10-26,880674609,148.0,Français English Español Italiano Deutsch,Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,Legendary Pictures Warner Bros. DC Entertainme...,United States of America,2012-07-16,1084939099,165.0,English,Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,Walt Disney Pictures,United States of America,2012-03-07,284139100,132.0,English,Released,"Lost in our world, found in another.",John Carter,6.1,2124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,220000,Action Crime Thriller,NaN,9367,united states–mexico barrier legs arms paper k...,es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,Columbia Pictures,Mexico United States of America,1992-09-04,2040920,81.0,Español,Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238
4799,9000,Comedy Romance,NaN,72766,,en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,,,2011-12-26,0,85.0,,Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5
4800,0,Comedy Drama Romance TV Movie,http://www.hallmarkchannel.com/signedsealeddel...,231617,date love at first sight narration investigati...,en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,Front Street Pictures Muse Entertainment Enter...,United States of America,2013-10-13,0,120.0,English,Released,NaN,"Signed, Sealed, Delivered",7.0,6
4801,0,,http://shanghaicalling.com/,126186,,en,Shanghai Calling,When ambitious New York attorney Sam is sent t...,0.857008,,United States of America China,2012-05-03,0,98.0,English,Released,A New Yorker in Shanghai,Shanghai Calling,5.7,7


In [35]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [36]:
data = data[['original_title', 'genres', 'popularity', 'runtime', 'vote_average', 'vote_count']]
data

,original_title,genres,popularity,runtime,vote_average,vote_count
0,Avatar,Action Adventure Fantasy Science Fiction,150.437577,162.0,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,139.082615,169.0,6.9,4500
2,Spectre,Action Adventure Crime,107.376788,148.0,6.3,4466
3,The Dark Knight Rises,Action Crime Drama Thriller,112.312950,165.0,7.6,9106
4,John Carter,Action Adventure Science Fiction,43.926995,132.0,6.1,2124
...,...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,14.269792,81.0,6.6,238
4799,Newlyweds,Comedy Romance,0.642552,85.0,5.9,5
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,1.444476,120.0,7.0,6
4801,Shanghai Calling,,0.857008,98.0,5.7,7


In [37]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_title  4803 non-null   object 
 1   genres          4803 non-null   object 
 2   popularity      4803 non-null   float64
 3   runtime         4801 non-null   float64
 4   vote_average    4803 non-null   float64
 5   vote_count      4803 non-null   int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 225.3+ KB


In [38]:
data = data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4801 entries, 0 to 4802
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_title  4801 non-null   object 
 1   genres          4801 non-null   object 
 2   popularity      4801 non-null   float64
 3   runtime         4801 non-null   float64
 4   vote_average    4801 non-null   float64
 5   vote_count      4801 non-null   int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 262.6+ KB


# 장르 컬럼의 텍스트를 숫자로 벡터화

In [39]:
from sklearn.feature_extraction.text import CountVectorizer 
# 문장을 단어 등장 횟수로 바꾸는 도구

In [40]:
cv = CountVectorizer(ngram_range=(1,2))
genres_vec = cv.fit_transform(data['genres'])
# fit: data['genres']전체를 훑어서 Action Adventure Fantasy와 같은 단어/단어쌍들 수집 후 각각에 번호 붙임
# transform: 각 행의 텍스트를 보고 숫자 벡터로 바꿈

In [41]:
print(len(cv.get_feature_names_out()))
# CountVectorizer가 만든 feature 목록 개수
# ex) ['action', 'adventure', 'comedy', 'fantasy', 'romance','action adventure', 'adventure fantasy', 'comedy romance']

276


In [42]:
genres_vec_df = pd.DataFrame(genres_vec)
genres_vec_df

,0
0,"(0, 0)\t1\n (0, 16)\t1\n (0, 124)\t1\n (0..."
1,"(0, 0)\t1\n (0, 16)\t1\n (0, 124)\t1\n (0..."
2,"(0, 0)\t1\n (0, 16)\t1\n (0, 1)\t1\n (0, ..."
3,"(0, 0)\t1\n (0, 64)\t1\n (0, 90)\t1\n (0,..."
4,"(0, 0)\t1\n (0, 16)\t1\n (0, 232)\t1\n (0..."
...,...
4796,"(0, 0)\t1\n (0, 64)\t1\n (0, 234)\t1\n (0..."
4797,"(0, 44)\t1\n (0, 214)\t1\n (0, 58)\t1"
4798,"(0, 90)\t1\n (0, 44)\t1\n (0, 214)\t1\n (..."
4799,


# 비슷한 장르를 찾기 위해서 코사인 유사도 구하기
* 코사인 유사도: 두 벡터가 얼마나 비슷한 방향을 가지는지 측정하는 방법

In [43]:
from sklearn.metrics.pairwise import cosine_similarity

In [44]:
genres_sim = cosine_similarity(genres_vec, genres_vec)
print(genres_sim.shape)

(4801, 4801)


In [45]:
genres_sim_df = pd.DataFrame(genres_sim) # 유사도는 1이 같은거고 0이 다른거임
genres_sim_df

,0,1,2,3,4,5,6,7,8,9,...,4791,4792,4793,4794,4795,4796,4797,4798,4799,4800
0,1.000000,0.596285,0.447214,0.125988,0.755929,0.596285,0.0,0.755929,0.447214,0.745356,...,0.000000,0.000000,0.000000,0.377964,0.000000,0.149071,0.0000,0.000000,0.0,0.0
1,0.596285,1.000000,0.400000,0.169031,0.338062,0.800000,0.0,0.338062,0.600000,0.800000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,0.0000,0.000000,0.0,0.0
2,0.447214,0.400000,1.000000,0.338062,0.507093,0.600000,0.0,0.507093,0.200000,0.600000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.400000,0.0000,0.000000,0.0,0.0
3,0.125988,0.169031,0.338062,1.000000,0.142857,0.169031,0.0,0.142857,0.000000,0.169031,...,0.377964,0.169031,0.377964,0.428571,0.218218,0.676123,0.0000,0.125988,0.0,0.0
4,0.755929,0.338062,0.507093,0.142857,1.000000,0.507093,0.0,1.000000,0.169031,0.507093,...,0.000000,0.000000,0.000000,0.428571,0.000000,0.169031,0.0000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,0.149071,0.200000,0.400000,0.676123,0.169031,0.200000,0.0,0.169031,0.000000,0.200000,...,0.000000,0.200000,0.000000,0.169031,0.258199,1.000000,0.0000,0.000000,0.0,0.0
4797,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.258199,0.000000,0.000000,0.000000,0.000000,1.0000,0.384900,0.0,0.0
4798,0.000000,0.000000,0.000000,0.125988,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.333333,0.149071,0.333333,0.125988,0.000000,0.000000,0.3849,1.000000,0.0,0.0
4799,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.0,0.0


In [52]:
sorted_genres_sim = genres_sim.argsort()[:,::-1] # 전체 행, 열 역순 정렬

In [53]:
sorted_genres_sim[:1]

array([[   0,   14,  870, ..., 3871, 2339, 4800]], dtype=int64)

In [83]:
data.iloc[14, :]

original_title                                Man of Steel
genres            Action Adventure Fantasy Science Fiction
popularity                                       99.398009
runtime                                              143.0
vote_average                                           6.5
vote_count                                            6359
Name: 14, dtype: object

# 영화 이름과 추천 받을 개수를 입력받아 출력하는 함수

In [63]:
searched_idx = data[data['original_title'] == 'Avatar'].index.values[0]
print("searched_idx", searched_idx)
recommanded_idx = sorted_genres_sim[searched_idx, :11]
recommanded_idx = recommanded_idx.tolist()
print(recommanded_idx)
if searched_idx in recommanded_idx:
    recommanded_idx.remove(searched_idx)
    print(recommanded_idx)
data.iloc[recommanded_idx, :]

searched_idx 0
[0, 14, 870, 46, 813, 3493, 1296, 1652, 419, 420, 1932]
[14, 870, 46, 813, 3493, 1296, 1652, 419, 420, 1932]


,original_title,genres,popularity,runtime,vote_average,vote_count
14,Man of Steel,Action Adventure Fantasy Science Fiction,99.398009,143.0,6.5,6359
870,Superman II,Action Adventure Fantasy Science Fiction,30.515175,127.0,6.5,629
46,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction,118.078691,131.0,7.5,6032
813,Superman,Action Adventure Fantasy Science Fiction,48.507081,143.0,6.9,1022
3494,Beastmaster 2: Through the Portal of Time,Action Adventure Fantasy Science Fiction,1.478505,107.0,4.6,17
1296,Superman III,Comedy Action Adventure Fantasy Science Fiction,22.164202,125.0,5.3,490
1652,Dragonball Evolution,Action Adventure Fantasy Science Fiction Thriller,21.677732,85.0,2.9,462
419,Jumper,Adventure Fantasy Science Fiction,21.218000,88.0,5.9,1799
420,Hellboy II: The Golden Army,Adventure Fantasy Science Fiction,58.579760,120.0,6.5,1527
1932,Sheena,Action Adventure Comedy Fantasy Science Fiction,4.020194,117.0,5.0,22


In [79]:
def find_sim_movie(data, sorted_genres_sim, title_name, top_n=11):
    searched_idx = data[data['original_title'] == title_name].index.values[0]
    print(searched_idx)
    recommanded_idx = sorted_genres_sim[searched_idx, :top_n+1]
    recommanded_idx = recommanded_idx.tolist()
    print(recommanded_idx)
    if searched_idx in recommanded_idx:
        recommanded_idx.remove(searched_idx)
    return data.loc[recommanded_idx, :]

In [80]:
data

,original_title,genres,popularity,runtime,vote_average,vote_count
0,Avatar,Action Adventure Fantasy Science Fiction,150.437577,162.0,7.2,11800
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,139.082615,169.0,6.9,4500
2,Spectre,Action Adventure Crime,107.376788,148.0,6.3,4466
3,The Dark Knight Rises,Action Crime Drama Thriller,112.312950,165.0,7.6,9106
4,John Carter,Action Adventure Science Fiction,43.926995,132.0,6.1,2124
...,...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,14.269792,81.0,6.6,238
4799,Newlyweds,Comedy Romance,0.642552,85.0,5.9,5
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,1.444476,120.0,7.0,6
4801,Shanghai Calling,,0.857008,98.0,5.7,7


In [81]:
find_sim_movie(data, sorted_genres_sim, 'Newlyweds', top_n=5)

4799
[4800, 1593, 1595, 1596, 1597, 1598]


,original_title,genres,popularity,runtime,vote_average,vote_count
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,1.444476,120.0,7.0,6
1593,The Three Stooges,Comedy,14.585968,92.0,4.9,140
1595,Glory Road,Drama,8.305085,118.0,7.2,144
1596,Sicario,Action Crime Drama Mystery Thriller,55.424027,121.0,7.2,2416
1597,Southpaw,Action Drama,65.364452,123.0,7.3,2067
1598,Drag Me to Hell,Horror Thriller,45.083509,99.0,6.2,966


# 영화이름 입력 받아서 비슷한 장르 영화 추천 받기

In [82]:
movie_name = input("영화 이름을 입력하세요: ")
find_sim_movie(data, sorted_genres_sim, movie_name, top_n=5)

영화 이름을 입력하세요: Spectre
2
[1740, 1542, 2, 969, 1082, 1076]


,original_title,genres,popularity,runtime,vote_average,vote_count
1740,Kick-Ass 2,Action Adventure Crime,40.286350,103.0,6.3,2224
1542,Speed,Action Adventure Crime,49.526736,116.0,6.8,1783
969,Assassins,Action Adventure Crime Thriller,23.073933,132.0,6.0,387
1082,16 Blocks,Action Adventure Crime Thriller,32.310220,105.0,6.2,661
1076,Cellular,Action Adventure Crime Thriller,24.459948,94.0,6.1,537
